In [1]:
# Celda 1 — SIEMPRE primero, antes que nada
import os
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import cv2, torch
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
torch.cuda.empty_cache()
print(f"VRAM libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")


VRAM libre: 15.8 GB


In [ ]:
# Celda 2 — Datos
import pandas as pd
from utils.train import get_adversarial_dataloaders
from utils.train.trainer_adversarial_v2 import train_model
from utils.models import Daowa_maadPrueba

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df_oxford   = pd.read_csv("labels/dataset_generated.csv")
df_ade20k   = pd.read_csv("labels/ade20k_adversarial_train.csv")
df_personas = pd.read_csv("labels/ade20k_person_negatives.csv")  

# 1. Cargar nuestro nuevo dataset súper difícil
df_fur = pd.read_csv("labels/hard_negatives_fur.csv")

# 2. Separar Perros (0) y Pura Textura (1)
df_fur_pos = df_fur[df_fur['is_adv'] == 0].copy()
df_fur_neg = df_fur[df_fur['is_adv'] == 1].copy()

# 3. OVERSAMPLING BRUTAL: Multiplicar para obligarlo a aprender la diferencia
# Un oversampling súper suave para no darle amnesia al modelo
df_fur_pos = pd.concat([df_fur_pos] * 5, ignore_index=True)  # De 3 pasamos a solo 15
df_fur_neg = pd.concat([df_fur_neg] * 3, ignore_index=True)  # De 15 pasamos a solo 45

# 4. Renombrar columnas para estandarizar
df_ade20k   = df_ade20k.rename(columns={"image": "file"})
df_personas = df_personas.rename(columns={"image": "file"})
df_fur_pos  = df_fur_pos.rename(columns={"image": "file"})
df_fur_neg  = df_fur_neg.rename(columns={"image": "file"})

if 'is_gold' in df_oxford.columns:
    df_oxford = df_oxford[df_oxford['is_gold'] == False]

# 5. INYECCIÓN AL DATASET BASE
df_oxford = pd.concat([df_oxford, df_fur_pos], ignore_index=True)      # Perros van con perros
df_personas = pd.concat([df_personas, df_fur_neg], ignore_index=True)  # Texturas van con personas

# 6. Generar loaders
loaders = get_adversarial_dataloaders(
    df_oxford=df_oxford,
    df_ade20k=df_ade20k,
    df_personas=df_personas,          
    oxford_dir="data/oxford",
    ade20k_dir="data/ADEChallengeData2016",
    batch_size=16,
    num_pos_per_batch=13,
    img_size=(384, 384),              # rápido para probar
    val_split=0.1,
    test_split=0.1,
)


ℹ️  Personas añadidas como hard negatives: 1739 train | 218 val | 217 test
✅ DataLoader adversarial listo — 5878 positivos | 9609 negativos | 452 batches/época (13P + 3N por batch)
✅ DataLoaders adversariales listos —
   Train : 5878 pos + 9609 neg
   Val   : 734 pos + 1202 neg
   Test  : 735 pos + 1200 neg


In [3]:
# Celda 3 — Modelo y entrenamiento
modelo = Daowa_maadPrueba(num_clases=1)

# Carga el mejor modelo adversarial 
# (Verifica si el tuyo es del día 12 o 13 de Mayo, lo dejé en 13 por si acaso)
peso_path = "Models/BestScore_Adv_2026-05-13_0.984.pth"
if not os.path.exists(peso_path):
    peso_path = "Models/BestScore_Adv_2026-05-12_0.984.pth"

print(f"Cargando pesos de: {peso_path}")
modelo.load_state_dict(torch.load(peso_path, map_location=device), strict=False)

# Usamos tu learning rate ultra bajo (5e-6) porque no queremos romper lo que ya aprendió
optimizer = torch.optim.AdamW(modelo.parameters(), lr=5e-6, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-7)

train_model(
    modelo=modelo,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    optimizador=optimizer,
    device=device,
    epochs=10,          # pocas para no sobrescribir lo aprendido
    patience=5,
    scheduler=scheduler,
    accum_steps=4,
    w_boundary=0.3,     # arranca más alto que antes (el modelo ya es estable)
)


Output()

Cargando pesos de: Models/BestScore_Adv_2026-05-13_0.984.pth
[trainer] 💾 Métricas por época → c:\Users\PC\Desktop\Abbadon prueba SAM\source\logs\history_adv_2026-05-14.csv


✓ Nuevo mejor IoU adversarial: 0.9986

Epoch 1: TrainLoss=0.9639 ValLoss=0.9821 | Acc_Pos=60.6% Acc_Neg=99.5% | IoU_Pos=0.6937 IoU_Neg=0.9986

Pesos → Dice=1.00 Boundary=0.10

...

Epoch 8: TrainLoss=0.4705 ValLoss=0.5726 | Acc_Pos=68.5% Acc_Neg=99.8% | IoU_Pos=0.6537 IoU_Neg=0.9987

Pesos → Dice=0.61 Boundary=0.80

Early stopping: 5 epochs sin mejora.

[trainer] ✅ Historial completo en: c:\Users\PC\Desktop\Abbadon prueba SAM\source\logs\history_adv_2026-05-14.csv


'c:\\Users\\PC\\Desktop\\Abbadon prueba SAM\\source\\logs\\history_adv_2026-05-14.csv'